# الدرس 13 - ذاكرة الوكيل مع رسوم المعرفة Cognee


## الإعداد

يوضح هذا الدفتر كيفية بناء **مساعد برمجي** ذكي ذا ذاكرة دائمة باستخدام رسوم المعرفة [**Cognee**](https://www.cognee.ai/) و**إطار عمل مايكروسوفت एजنت** (MAF).

يقوم Cognee بتحويل النصوص غير المنظمة إلى رسم معرفة منظم وقابل للاستعلام مدعوم بتضمينات متجهة — مما يمنح الوكيل ذاكرة طويلة الأمد غنية وواعية بالعلاقات.

### ما ستتعلمه
1. **بناء رسوم المعرفة**: تحويل ملفات تعريف المطورين وأفضل الممارسات إلى معرفة منظمة وقابلة للاستعلام.
2. **دمج Cognee مع MAF**: استخدام دوال `@tool` لتمكين وكيل MAF من استعلام رسم معرفة Cognee.
3. **محادثات واعية للجلسة**: الحفاظ على السياق عبر أسئلة متعددة في نفس الجلسة.
4. **ذاكرة طويلة الأمد**: حفظ المعرفة المهمة عبر الجلسات واستردادها في محادثات جديدة.

### المتطلبات الأساسية
- بايثون 3.9+
- تشغيل Redis محليًا (`docker run -d -p 6379:6379 redis`) لإدارة الجلسات
- مفتاح API لنموذج لغة كبير (LLM) مثل OpenAI — ضع `LLM_API_KEY` في `.env`
- `CACHING=true` في `.env` (مطلوب لجلسات Cognee)
- مشروع Microsoft Foundry مع نموذج دردشة منشور
- `AZURE_AI_PROJECT_ENDPOINT` و `AZURE_AI_MODEL_DEPLOYMENT_NAME` في `.env`
- تم المصادقة على Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## أنواع ذاكرة الوكيل

تستكشف هذه الدفتر نفس الأنواع الثلاثة للذاكرة من دفتر الدرس 13 الرئيسي، ولكنها تستخدم Cognee كقاعدة لذاكرة طويلة الأمد:

| نوع الذاكرة | الآلية | مدة الحياة |
|---|---|---|
| **عاملة** | `agent.create_session()` (MAF) | محادثة واحدة |
| **قصيرة الأمد** | ذاكرة جلسة Cognee المؤقتة (Redis) | جلسة واحدة |
| **طويلة الأمد** | رسم المعرفة Cognee + المتجهات | دائمة |

### بنية ذاكرة Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## إعداد تخزين Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## الجزء 1 — بناء قاعدة المعرفة

نحن نستوعب ثلاثة أنواع من البيانات لإنشاء قاعدة معرفة شاملة لمساعدنا البرمجي:

1. **ملف المطور** — الخبرة الشخصية والخلفية التقنية
2. **أفضل ممارسات بايثون** — زِن بايثون مع إرشادات عملية
3. **المحادثات التاريخية** — جلسات الأسئلة والأجوبة الماضية بين المطورين والمساعدين الذكاء الاصطناعي


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## تصوُّر رسم المعرفة

يمكن لـ Cognee عرض تصور HTML تفاعلي للكيانات والعلاقات التي استخرجها.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## إثراء الذاكرة باستخدام Memify

يقوم `memify()` بتحليل رسم المعرفة وتوليد قواعد ذكية — لتحديد الأنماط وأفضل الممارسات والعلاقات بين المفاهيم.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## الجزء 2 — وكيل MAF مع أدوات Cognee

الآن نقوم بإنشاء وكيل MAF يمكنه الاستعلام من خلال رسم المعرفة الخاص بـ Cognee عبر دوال `@tool`. هذا يتيح للوكيل الاستفادة من القوة الكاملة للبحث الدلالي الواعي بالرسم مع الحفاظ على سياق المحادثة عبر الجلسات.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## الذاكرة العاملة مع الجلسات

يوفر `AgentSession` (الذي يتم إنشاؤه عبر `agent.create_session()`) ذاكرة عاملة داخل الجلسة. يمكن للوكيل الرجوع إلى الرسائل السابقة مع الاستعلام أيضًا من خلال مخطط المعرفة طويلة الأمد الخاص بـ Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## جلسة جديدة — الذاكرة طويلة الأمد تستمر

بدء جلسة جديدة يمسح الذاكرة العاملة، لكن رسم المعرفة Cognee لا يزال متاحًا. يمكن للوكيل استرجاع نفس المعرفة طويلة الأمد في محادثة جديدة تمامًا.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## الملخص

في هذا الدفتر أنشأت مساعد ترميز يجمع بين **ذاكرة العمل لـ MAF** (`agent.create_session()`) و**مخطط المعرفة طويل الأمد لـ Cognee**.

### ما تعلمته
1. **بناء مخطط المعرفة**: يقوم Cognee باستخلاص النص غير المنظم وبناء مخطط + ذاكرة متجهة.
2. **إثراء المخطط باستخدام memify**: حقائق مشتقة وعلاقات أكثر ثراءً فوق المخطط الحالي.
3. **تكامل MAF + Cognee**: تتيح دوال `@tool` لوكيل MAF الاستعلام من مخطط Cognee بطريقة طبيعية.
4. **ذاكرة العمل + الذاكرة طويلة الأمد**: يوفر `AgentSession` (عن طريق `agent.create_session()`) سياق الجلسة بينما يوفر Cognee المعرفة المستمرة.
5. **البحث المصفى باستخدام NodeSets**: استهداف مجموعات فرعية محددة من مخطط المعرفة (مثل المبادئ فقط).

### النقاط الرئيسية
- يحول **Cognee** النص الخام إلى ذاكرة منظمة وواعية بالعلاقات — أقوى من متجر المتجهات المسطحة.
- تربط **دوال `@tool`** بين وكلاء MAF وأنظمة المعرفة الخارجية بشكل نظيف.
- تحافظ **`AgentSession`** (من خلال `agent.create_session()`) على سياق كل محادثة منفصلًا عن المعرفة طويلة الأمد.
- يخدم نفس مخطط المعرفة جلسات ووكلاء متعددة.

### التطبيقات الواقعية
- **مساعدو المطورين**: مراجعة الشيفرة، تحليل الحوادث، مساعدو الهندسة المعمارية
- **مساعدو خدمة العملاء**: وكلاء الدعم عبر مستندات المنتج، الأسئلة الشائعة، وملاحظات إدارة علاقات العملاء
- **مساعدو الخبراء الداخليين**: المساعدون في السياسات، الشؤون القانونية، أو الأمان الذين يستدلون على الإرشادات
- **طبقات بيانات موحدة**: دمج البيانات المنظمة وغير المنظمة في مخطط قابل للاستعلام

### الخطوات التالية
- تجربة الوعي الزمني في Cognee
- تعريف أونتولوجيا OWL لجودة المخطط الخاصة بالمجال
- إضافة حلقات تغذية راجعة من المستخدم لتحسين الاسترجاع مع الوقت
- التوسع إلى أنظمة متعددة الوكلاء تشترك في نفس طبقة ذاكرة Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
